# WebsiteSearchTool

`WebsiteSearchTool` is a RAG (Retrieval-Augmented Generation) tool in the `crewai-tools` package. It performs semantic search over website content by scraping a URL, indexing it in a vector store, and querying for relevant chunks.


## Location

| Item | Path |
|------|------|
| Implementation | `lib/crewai-tools/src/crewai_tools/tools/website_search/website_search_tool.py` |
| Import | `from crewai_tools import WebsiteSearchTool` |
| Edge docs | `docs/edge/en/tools/search-research/websitesearchtool.mdx` |
| RAG base docs | `docs/edge/en/tools/ai-ml/ragtool.mdx` |

## Class Hierarchy

<br>

```
BaseTool
  └── RagTool
        └── WebsiteSearchTool
```

Most configuration options are inherited from `RagTool`.


## How It Works

1. Website URLs are validated with `validate_url()`.
2. Content is scraped and chunked using `DataType.WEBSITE` (`WebPageLoader` + `WebsiteChunker`).
3. Chunks are stored in a vector database (ChromaDB by default).
4. Queries return the most similar chunks above the similarity threshold.

## Installation

<br>

```shell
pip install 'crewai[tools]'
```


## Usage Modes


### Any website (agent supplies URL at runtime)

<br>

```python
from crewai_tools import WebsiteSearchTool

tool = WebsiteSearchTool()
# Agent must call with both search_query and website
```

Uses `WebsiteSearchToolSchema`, which requires:
- `search_query` — the semantic search query
- `website` — a valid website URL to search


### Fixed website (indexed at initialization)

<br>

```python
from crewai_tools import WebsiteSearchTool

tool = WebsiteSearchTool(website="https://example.com")
# Agent only needs search_query
```

When `website` is provided at init, the tool:
- Immediately scrapes and indexes that URL via `add(website, data_type=DataType.WEBSITE)`
- Updates its description to reference the fixed URL
- Switches to `FixedWebsiteSearchToolSchema` (only `search_query` is required)

## Configuration Options


### 1. Website-Specific (Constructor)

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `website` | `str \| None` | `None` | Optional URL to pin the tool to a single site |


### 2. Inherited from RagTool (Constructor `**kwargs`)

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `summarize` | `bool` | `False` | Whether to summarize retrieved content |
| `similarity_threshold` | `float` | `0.6` | Minimum similarity score for returned results |
| `limit` | `int` | `5` | Maximum number of chunks to return |
| `collection_name` | `str` | `"rag_tool_collection"` | Vector database collection name |
| `adapter` | `Adapter` | auto | Custom RAG adapter (mainly used in tests) |
| `config` | `RagToolConfig` | `{}` | RAG backend configuration (see below) |

Example:

```python
tool = WebsiteSearchTool(
    website="https://example.com",
    summarize=True,
    similarity_threshold=0.7,
    limit=10,
    collection_name="my_website_index",
)
```



### 3. `config` Dict (`RagToolConfig`)

Defined in `lib/crewai-tools/src/crewai_tools/tools/rag/types.py`:

```python
config = {
    "embedding_model": { ... },  # ProviderSpec (optional)
    "vectordb": { ... },         # VectorDbConfig (optional)
}
```

Programmatic config takes precedence over environment variables.

#### Embedding Model

Customize the embedding provider and model. Supported providers include:

- `openai`
- `ollama`
- `azure`
- `google-generativeai`
- `google-vertex`
- `cohere`
- `voyageai`
- `huggingface`
- `amazon-bedrock`
- `jina`
- `instructor`
- `sentence-transformer`
- `onnx`
- `openclip`
- `text2vec`
- `roboflow`
- `watsonx`
- `custom`

Example with OpenAI:

```python
tool = WebsiteSearchTool(
    website="https://example.com",
    config={
        "embedding_model": {
            "provider": "openai",
            "config": {"model": "text-embedding-3-small"},
        },
    },
)
```

Example with Ollama:

```python
tool = WebsiteSearchTool(
    config={
        "embedding_model": {
            "provider": "ollama",
            "config": {"model": "nomic-embed-text"},
        },
    },
)
```

See `docs/edge/en/tools/ai-ml/ragtool.mdx` for full provider-specific config examples.

#### Vector Database

Default provider is `chromadb`. Also supported: `qdrant`.

Example with Qdrant:

```python
tool = WebsiteSearchTool(
    config={
        "vectordb": {
            "provider": "qdrant",
            "config": {"collection_name": "my-collection"},
        },
        "embedding_model": {
            "provider": "openai",
            "config": {"model": "text-embedding-3-small"},
        },
    },
)
```

Example with ChromaDB (explicit):

```python
tool = WebsiteSearchTool(
    config={
        "vectordb": {
            "provider": "chromadb",
            "config": {},
        },
        "embedding_model": {
            "provider": "openai",
            "config": {"model": "text-embedding-3-small"},
        },
    },
)
```



### 4. Runtime Parameters (`_run` / Agent Invocation)

| Parameter | Required | Description |
|-----------|----------|-------------|
| `search_query` | Yes | Semantic search query |
| `website` | Only if not fixed at init | URL to scrape and search |
| `similarity_threshold` | No | Override the default threshold for this call |
| `limit` | No | Override the default result count for this call |

If `website` is passed at runtime (and was not fixed at init), it triggers `add()` to scrape and index that URL before querying.


In [0]:
%skip

class WebsiteSearchTool(RagTool):
    name: str = "Search in a specific website"
    description: str = "A tool that can be used to semantic search a query from a specific URL content."
    args_schema: type[BaseModel] = WebsiteSearchToolSchema

    def __init__(self, website: str | None = None, **kwargs: Any) -> None:
        super().__init__(**kwargs)
        if website is not None:
            self.add(website)
            self.description = f"A tool that can be used to semantic search a query from {website} website content."
            self.args_schema = FixedWebsiteSearchToolSchema
            self._generate_description()

    def add(self, website: str) -> None:
        website = validate_url(website)
        super().add(website, data_type=DataType.WEBSITE)

    def _run(
        self,
        search_query: str,
        website: str | None = None,
        similarity_threshold: float | None = None,
        limit: int | None = None,
    ) -> str:
        if website is not None:
            self.add(website)
        return super()._run(
            query=search_query,
            similarity_threshold=similarity_threshold,
            limit=limit,
        )


## Agent Integration

<br>

```python
from crewai import Agent
from crewai_tools import WebsiteSearchTool

search_tool = WebsiteSearchTool(website="https://docs.crewai.com")

agent = Agent(
    role="Researcher",
    goal="Find information on CrewAI documentation",
    backstory="Expert at web research",
    tools=[search_tool],
)
```

## Input Schemas

### `WebsiteSearchToolSchema` (dynamic website mode)

Used when no `website` is set at initialization.

| Field | Required | Description |
|-------|----------|-------------|
| `search_query` | Yes | Mandatory search query |
| `website` | Yes | Mandatory valid website URL |

### `FixedWebsiteSearchToolSchema` (fixed website mode)

Used when `website` is set at initialization.

| Field | Required | Description |
|-------|----------|-------------|
| `search_query` | Yes | Mandatory search query |


## Important Notes

1. **Outdated docs warning:** The README in this directory and `docs/edge/en/tools/search-research/websitesearchtool.mdx` show a `config` format with `llm` and `embedder` keys. The actual
`RagTool` API uses `embedding_model` and `vectordb`. There is no `llm` config on this tool.

2. **Indexing behavior:** Each new `website` passed at runtime triggers `add()`, which scrapes and indexes that URL into the vector store.

3. **Security:** URLs are validated via `validate_url()` before scraping to prevent unsafe requests.

4. **Default embeddings:** When no `embedding_model` is configured, the tool uses CrewAI's default RAG client configuration (typically OpenAI embeddings via environment variables).

5. **Return format:** `_run` returns a string prefixed with `"Relevant Content:\n"` followed by the matched chunks.


## Related Files

- `lib/crewai-tools/src/crewai_tools/tools/rag/rag_tool.py` — base RAG tool
- `lib/crewai-tools/src/crewai_tools/tools/rag/types.py` — `RagToolConfig`, `VectorDbConfig`
- `lib/crewai-tools/src/crewai_tools/adapters/crewai_rag_adapter.py` — default adapter
- `lib/crewai-tools/tests/tools/test_search_tools.py` — unit tests for `WebsiteSearchTool`